# E5 — KV cache e latenza

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emanbuc/corso_foundation_tabular/blob/main/notebooks/usare-tfm/E5_kv_cache_latenza.ipynb)

**Corso 1 · Usare un TFM — Modulo 2 (laboratorio, 10 min)**

In un TFM il costo si paga in inferenza: a ogni `predict` il modello rielabora tutto il contesto. La KV cache salva in `fit` i calcoli che dipendono solo dal contesto, per riusarli a ogni chiamata.

**Idea da far emergere:** il tempo di predizione con e senza cache del contesto, e il costo della cache in memoria.

- **Dati:** Adult (OpenML), lo stesso dataset di E4.
- **Scenario:** un contesto fisso da 5.000 righe e molte richieste piccole, come in un servizio che risponde a una richiesta alla volta.

Serve la GPU. Senza GPU ridurre `N_CONTESTO` a 1.000; la memoria non viene misurata.

## 0. Setup

In [ ]:
%pip install -q tabicl==2.2.0

In [ ]:
import gc
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from tabicl import TabICLClassifier

GPU = torch.cuda.is_available()
print("GPU disponibile:", GPU)

## 1. Dati e scenario

In [ ]:
N_CONTESTO = 5000          # righe di contesto
N_CHIAMATE = 20            # numero di richieste
RIGHE_PER_CHIAMATA = 100   # righe di test per richiesta

adult = fetch_openml("adult", version=2, as_frame=True)
X = adult.data
y = adult.target.astype(str).str.contains(">50K").astype(int).to_numpy()

X_ctx, X_te, y_ctx, y_te = train_test_split(
    X, y, train_size=N_CONTESTO, test_size=N_CHIAMATE * RIGHE_PER_CHIAMATA,
    stratify=y, random_state=0)
print("Contesto:", X_ctx.shape, " Test:", X_te.shape)

## 2. Tre configurazioni

Il parametro `kv_cache` di `tabicl` ha tre valori:

| Valore | Cosa salva in `fit` |
| --- | --- |
| `False` | Niente: ogni `predict` rielabora il contesto |
| `True` | Le proiezioni chiave-valore di tutti gli strati: inferenza più veloce, più memoria |
| `"repr"` | Solo una parte dei calcoli: meno memoria, ma a ogni `predict` il transformer finale va rieseguito |

Per ogni configurazione si misurano il tempo di `fit`, il tempo di ogni `predict_proba` e la memoria GPU.

In [ ]:
def misura(kv_cache):
    if GPU:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    inizio = time.perf_counter()
    modello = TabICLClassifier(kv_cache=kv_cache).fit(X_ctx, y_ctx)
    if GPU:
        torch.cuda.synchronize()
    tempo_fit = time.perf_counter() - inizio
    memoria_dopo_fit = torch.cuda.memory_allocated() / 2**30 if GPU else float("nan")

    tempi, probabilita = [], []
    for i in range(N_CHIAMATE):
        blocco = X_te.iloc[i * RIGHE_PER_CHIAMATA:(i + 1) * RIGHE_PER_CHIAMATA]
        inizio = time.perf_counter()
        probabilita.append(modello.predict_proba(blocco)[:, 1])
        if GPU:
            torch.cuda.synchronize()
        tempi.append(time.perf_counter() - inizio)
    memoria_picco = torch.cuda.max_memory_allocated() / 2**30 if GPU else float("nan")

    del modello
    gc.collect()
    return {
        "fit (s)": tempo_fit,
        "predict, mediana (s)": float(np.median(tempi)),
        "memoria dopo fit (GB)": memoria_dopo_fit,
        "memoria di picco (GB)": memoria_picco,
        "tempi": tempi,
        "probabilita": np.concatenate(probabilita),
    }

La prima chiamata scarica i pesi e inizializza la GPU: si fa a vuoto, per non sporcare i tempi.

In [ ]:
_ = TabICLClassifier().fit(X_ctx.iloc[:200], y_ctx[:200]).predict_proba(X_te.iloc[:10])

In [ ]:
CONFIGURAZIONI = {"senza cache": False, "kv_cache=True": True, "kv_cache='repr'": "repr"}

misure = {nome: misura(valore) for nome, valore in CONFIGURAZIONI.items()}
tabella = pd.DataFrame(misure).T.drop(columns=["tempi", "probabilita"]).astype(float)
tabella.round(3)

### La cache cambia le predizioni?

Non dovrebbe: è lo stesso calcolo, fatto in due momenti diversi. Si controlla la differenza massima tra le probabilità.

In [ ]:
riferimento = misure["senza cache"]["probabilita"]
for nome in list(CONFIGURAZIONI)[1:]:
    differenza = np.abs(misure[nome]["probabilita"] - riferimento).max()
    print(f"{nome:16s} differenza massima rispetto a 'senza cache': {differenza:.4f}")

## 3. Quando conviene la cache?

**Grafico da commentare:** il tempo cumulato al crescere del numero di richieste. Ogni curva parte dal tempo di `fit` e sale di un gradino a ogni `predict`.

La cache costa in `fit` e fa risparmiare a ogni `predict`: conviene quando le curve si incrociano.

In [ ]:
fig, (sx, dx) = plt.subplots(1, 2, figsize=(11, 4))
for nome, m in misure.items():
    cumulato = m["fit (s)"] + np.concatenate([[0.0], np.cumsum(m["tempi"])])
    sx.plot(range(N_CHIAMATE + 1), cumulato, marker=".", label=nome)
sx.set_xticks(range(0, N_CHIAMATE + 1, 5))
sx.set_xlabel("numero di richieste servite")
sx.set_ylabel("tempo cumulato (s)")
sx.set_title("Tempo: fit + richieste")
sx.legend()

tabella[["memoria dopo fit (GB)", "memoria di picco (GB)"]].plot.bar(ax=dx, rot=0)
dx.set_ylabel("memoria GPU (GB)")
dx.set_title("Memoria")
plt.tight_layout()
plt.show()

## Da osservare

- Dopo quante richieste la cache ha ripagato il tempo in più speso in `fit`?
- Quanta memoria GPU resta occupata dopo `fit` nelle tre configurazioni?
- `"repr"` è un buon compromesso in questo scenario?

Domande per la discussione:

- In quale di questi casi usereste la cache: una predizione in blocco su 100.000 righe, oppure un servizio che risponde a una richiesta alla volta?
- Cosa succede a tempo e memoria della cache se il contesto passa da 5.000 a 10.000 righe? Si può provare cambiando `N_CONTESTO`.